# RQ3 — Persistence Prediction (views-inclusive)
Clean pipeline. Run top to bottom (Restart & Run All).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH = Path(r"C:\Users\user\cross_platform_analysis\4 - ER unified\finalData_with_er_yt_with_views.csv")
df = pd.read_csv(DATA_PATH, low_memory=False)

# Filter to 2022 only
df22 = df[df['_year'] == 2022].copy()
df22 = df22[df22['er_pct'].notna() & df22['followers'].notna()]

# Check month distribution
print("Month distribution:")
print(df22['_month'].value_counts())
print(f"\nPlatforms: {df22['_platform'].unique()}")
print(f"\nTotal records: {len(df22)}")


In [ ]:
# Separate December (target) from training months
TRAIN_MONTHS = ['march', 'june', 'september', 'november']
TEST_MONTH = 'december'

train = df22[df22['_month'].isin(TRAIN_MONTHS)].copy()
test_ids = set(df22[df22['_month'] == TEST_MONTH]['uniqueId'].dropna())

# Compute per-creator features from training months
def compute_features(group):
    er_vals = group['er_pct'].values
    fol_vals = group['followers'].values
    months = group['_month'].values
    
    mean_er = np.mean(er_vals)
    std_er = np.std(er_vals)
    cv_er = std_er / mean_er if mean_er > 0 else np.nan
    mean_followers = np.mean(fol_vals)
    n_months = len(group)
    
    # ER trend: slope via simple linear regression if >= 2 appearances
    month_order = {'march': 1, 'june': 2, 'september': 3, 'november': 4}
    if n_months >= 2:
        x = np.array([month_order[m] for m in months])
        er_trend = np.polyfit(x, er_vals, 1)[0]
    else:
        er_trend = np.nan
    
    return pd.Series({
        'mean_er': mean_er,
        'std_er': std_er,
        'cv_er': cv_er,
        'mean_followers': mean_followers,
        'n_months': n_months,
        'er_trend': er_trend,
        'platform': group['_platform'].iloc[0],
        'category': group['category_unified'].iloc[0]
    })

features = (
    train.dropna(subset=['uniqueId'])
         .groupby('uniqueId')
         .apply(compute_features)
         .reset_index()
)

# Add target: did creator appear in December?
features['persisted'] = features['uniqueId'].isin(test_ids).astype(int)

print(f"Total creators with training data: {len(features)}")
print(f"\nPersistence rate by platform:")
print(features.groupby('platform')['persisted'].mean().round(3))
print(f"\nMissing values:\n{features.isnull().sum()}")

In [ ]:
# Fix missing values
features['cv_er'] = features['cv_er'].fillna(0)
features['er_trend'] = features['er_trend'].fillna(0)

# Check category coverage per platform
print("Category coverage per platform:")
print(features.groupby('platform')['category'].apply(
    lambda x: x.notna().mean()
).round(3))

# Create has_category binary flag
features['has_category'] = features['category'].notna().astype(int)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Final feature set — no category, keep has_category
feature_cols = ['mean_er', 'std_er', 'cv_er', 'mean_followers', 
                'n_months', 'er_trend', 'has_category']

# Encode platform
le = LabelEncoder()
features['platform_enc'] = le.fit_transform(features['platform'])
feature_cols.append('platform_enc')

# Run per platform and pooled
print("Feature matrix shape:", features[feature_cols].shape)
print("\nClass distribution per platform:")
for platform in ['instagram', 'tiktok', 'youtube']:
    sub = features[features['platform'] == platform]
    n_pos = sub['persisted'].sum()
    n_neg = len(sub) - n_pos
    print(f"{platform:12s} | n={len(sub)} | persisted={n_pos} ({n_pos/len(sub)*100:.1f}%) | dropout={n_neg} ({n_neg/len(sub)*100:.1f}%)")
    

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score, 
                             confusion_matrix, classification_report)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

PLATFORMS = ['instagram', 'tiktok', 'youtube']
FEATURE_COLS = ['mean_er', 'std_er', 'cv_er', 'mean_followers',
                'n_months', 'er_trend', 'has_category']

results = []

for platform in PLATFORMS:
    print(f"\n{'='*55}")
    print(f"PLATFORM: {platform.upper()}")
    print(f"{'='*55}")
    
    sub = features[features['platform'] == platform].copy()
    X = sub[FEATURE_COLS].values
    y = sub['persisted'].values
    
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    scale_pos = n_neg / n_pos
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    models = {
        'Logistic Regression': LogisticRegression(
            class_weight='balanced', random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(
            class_weight='balanced', n_estimators=500, random_state=42),
        'XGBoost': XGBClassifier(
            scale_pos_weight=scale_pos, n_estimators=500,
            random_state=42, eval_metric='logloss', verbosity=0),
        'SVM': SVC(
            class_weight='balanced', kernel='rbf',
            probability=True, random_state=42)
    }
    
    for name, model in models.items():
        auc  = cross_val_score(model, X_scaled, y, cv=cv,
                               scoring='roc_auc').mean()
        f1   = cross_val_score(model, X_scaled, y, cv=cv,
                               scoring='f1').mean()
        acc  = cross_val_score(model, X_scaled, y, cv=cv,
                               scoring='accuracy').mean()
        
        print(f"\n{name}")
        print(f"  AUC-ROC  = {auc:.3f}")
        print(f"  F1       = {f1:.3f}")
        print(f"  Accuracy = {acc:.3f}")
        
        results.append({
            'platform': platform, 'model': name,
            'auc': auc, 'f1': f1, 'accuracy': acc
        })

results_df = pd.DataFrame(results)
print("\n\n── Summary Table ──")
print(results_df.pivot_table(
    index='model', columns='platform', values='auc'
).round(3))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

# ── Plot 1: AUC heatmap ───────────────────────────────────────────────────────
pivot = results_df.pivot_table(index='model', columns='platform', values='auc')
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0.6, vmax=0.9)
ax.set_xticks(range(len(pivot.columns)))
ax.set_yticks(range(len(pivot.index)))
ax.set_xticklabels([c.capitalize() for c in pivot.columns], fontsize=11)
ax.set_yticklabels(pivot.index, fontsize=11)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i,j]:.3f}",
                ha='center', va='center', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, label='AUC-ROC')
ax.set_title('AUC-ROC by Model and Platform\n(5-fold Stratified CV)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_PATH / 'ml_auc_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Plot 1 saved")

# ── Plot 2: ROC curves per platform ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
COLORS_ML = {
    'Logistic Regression': '#E1306C',
    'Random Forest': '#010101',
    'XGBoost': '#FF0000',
    'SVM': '#4169E1'
}

for ax, platform in zip(axes, PLATFORMS):
    sub = features[features['platform'] == platform].copy()
    X = sub[FEATURE_COLS].values
    y = sub['persisted'].values
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    models = {
        'Logistic Regression': LogisticRegression(
            class_weight='balanced', random_state=42, max_iter=1000),
        'Random Forest': RandomForestClassifier(
            class_weight='balanced', n_estimators=500, random_state=42),
        'XGBoost': XGBClassifier(
            scale_pos_weight=n_neg/n_pos, n_estimators=500,
            random_state=42, eval_metric='logloss', verbosity=0),
        'SVM': SVC(
            class_weight='balanced', kernel='rbf',
            probability=True, random_state=42)
    }
    
    for name, model in models.items():
        tprs, aucs = [], []
        mean_fpr = np.linspace(0, 1, 100)
        for train_idx, test_idx in cv.split(X_scaled, y):
            model.fit(X_scaled[train_idx], y[train_idx])
            proba = model.predict_proba(X_scaled[test_idx])[:, 1]
            fpr, tpr, _ = roc_curve(y[test_idx], proba)
            tprs.append(np.interp(mean_fpr, fpr, tpr))
            aucs.append(auc(fpr, tpr))
        mean_tpr = np.mean(tprs, axis=0)
        mean_auc = np.mean(aucs)
        ax.plot(mean_fpr, mean_tpr, color=COLORS_ML[name],
                label=f"{name} (AUC={mean_auc:.3f})", linewidth=2)
    
    ax.plot([0,1], [0,1], 'k--', linewidth=1, label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(platform.capitalize(), fontweight='bold', fontsize=13)
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('ROC Curves by Platform (5-fold Stratified CV)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_PATH / 'ml_roc_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print("✅ Plot 2 saved")

In [ ]:
# ── SHAP beeswarm: feature importance per platform (views-inclusive) ──
import shap
import matplotlib.pyplot as plt
from pathlib import Path

OUT_PATH = Path(r"C:\Users\user\cross_platform_analysis\8 - rq3\plots")

for platform in PLATFORMS:
    sub = features[features['platform'] == platform].copy()
    X = sub[FEATURE_COLS].values
    y = sub['persisted'].values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    rf = RandomForestClassifier(class_weight='balanced', n_estimators=500, random_state=42)
    rf.fit(X_scaled, y)

    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_scaled)
    # version-safe: list (old API) vs 3-D array (new API)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values[:, :, 1]

    plt.figure(figsize=(8, 5))
    shap.summary_plot(sv, X_scaled, feature_names=FEATURE_COLS, show=False)
    plt.title(platform.capitalize(), fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_PATH / f'shap_{platform}.png', bbox_inches='tight', dpi=150)
    plt.close()
    print(f"OK  shap_{platform}.png saved")
